[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/Deep_Variation_Information_Bottleneck_training/blob/master/ex006_mnist_nn_adv.ipynb)

> **実行する前に，ランタイムを GPU に変更すること．**  
> Google Colab ではメニューの **ランタイム → ランタイムのタイプを変更** を開き，ハードウェア アクセラレータを **GPU** に設定して保存する．CPU のままだと学習に時間がかかる．

# 実験006: 全結合 NN による MNIST の敵対的学習（ノイズ最適化）

本ノートブックでは，実験001と同じ全結合ニューラルネットワークで MNIST を分類する．ただし学習時に，入力へ加えるノイズ $\delta$ を誤差が大きくなる方向へ最適化する．

この講座では，次の順でプログラムを積み上げる．

1. NN による MNIST 分類（交差エントロピー誤差）
2. Autoencoder による MNIST の再構成
3. Variational Autoencoder による MNIST の再構成
4. Deep Variational Information Bottleneck による MNIST 分類
5. 2 次元特徴マップの可視化
6. **本実験**: 敵対的学習（ノイズ最適化）
7. Deep VIB + Brier スコアによる MNIST 分類
8. Deep VIB + クラス間重み付き CCE による MNIST 分類

学習ループの分割は実験001と同じである．本実験での変更点は，学習時に入力ノイズを PGD で最適化し，敵対的 Accuracy を評価することである．


## 1. ライブラリ


In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


## 2. 再現性とログ


In [ ]:
def set_seed(seed=0):
    """実験の再現性を担保するため，乱数シードを固定する．

    Args:
        seed (int): 固定するシード値．デフォルトは 0．

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)


class ResultLogger:
    """学習中の指標（Loss，Accuracy など）を記録し，JSON として保存・読み込みするクラス．

    Args:
        target_path (str | None): 既存ログ JSON のパス．指定時は初期化時に読み込む．

    Returns:
        なし（インスタンスを生成する）
    """

    def __init__(self, target_path=None):
        self.names = None
        self.history = {}
        if target_path:
            self.load(target_path)

    def set_names(self, *names):
        """記録する指標名を登録する．

        Args:
            *names (str): 指標名．例: "train_loss", "train_acc"．

        Returns:
            None
        """
        if self.names:
            raise RuntimeError("指標名はすでに登録されている．")
        self.names = list(names)
        for name in self.names:
            if name not in self.history:
                self.history[name] = []

    def __call__(self, *values):
        """1 エポック分の指標値を履歴へ追加する．

        Args:
            *values (float | int): set_names で登録した順の値．

        Returns:
            None
        """
        if self.names is None:
            raise RuntimeError("先に set_names で指標名を登録すること．")
        if len(values) != len(self.names):
            raise RuntimeError("値の数が登録済み指標名の数と一致しません．")
        for name, value in zip(self.names, values):
            self.history[name].append(value)

    def save(self, target_path):
        """履歴を JSON ファイルとして保存する．

        Args:
            target_path (str): 保存先パス．

        Returns:
            None
        """
        with open(target_path, "w") as f:
            json.dump(self.history, f, indent=4)

    def load(self, target_path):
        """JSON ファイルから履歴を読み込む．

        Args:
            target_path (str): 読み込む JSON のパス．

        Returns:
            None
        """
        with open(target_path, "r") as f:
            data = json.load(f)
        self.history = data
        self.names = list(data.keys())

    def __getitem__(self, key):
        """指定した指標の履歴リストを取得する．

        Args:
            key (str): 指標名．

        Returns:
            history (list): 指標の履歴．存在しない場合は空リスト．
        """
        return self.history.get(key, [])


## 3. 実験設定

`outputs/ex006_mnist_nn_adv/{最適化手法}/{学習率}_{バッチサイズ}/{seed}/`

本実験はサンプルのため，シードは 1 つ（`seed=0`）とする．$\epsilon$ はノイズの $L_\infty$ 半径，`ADV_STEPS` はノイズ最適化の反復回数である．


In [ ]:
EXPERIMENT_NAME = "ex006_mnist_nn_adv"

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == EXPERIMENT_NAME:
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATASET_DIR = PROJECT_ROOT / "datasets" / EXPERIMENT_NAME / "standard"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / EXPERIMENT_NAME

EPOCHS = 10
SEEDS = [0]
OPTIMIZER_NAMES = ["Adam"]
LEARNING_RATES = [0.001]
BATCH_SIZES = [128]
EPSILON = 0.1
ADV_STEPS = 5
ADV_STEP_SIZE = 0.025

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATASET_DIR  : {DATASET_DIR}")
print(f"OUTPUT_ROOT  : {OUTPUT_ROOT}")
print(f"EPSILON      : {EPSILON}")
print(f"ADV_STEPS    : {ADV_STEPS}")
print(f"ADV_STEP_SIZE: {ADV_STEP_SIZE}")


## 4. モデルの定義

実験001と同じ 3 層の全結合分類器である．入力画像を 784 次元に平坦化し，784 -> 256 -> 128 -> 10 の線形変換でクラスロジットを出力する．


In [ ]:
class MNISTClassifier(nn.Module):
    """MNIST を 10 クラス分類する全結合ニューラルネットワーク．

    入力画像を 784 次元に平坦化したのち，784 -> 256 -> 128 -> 10 の線形変換を行う．

    Args:
        なし（層の次元はクラス内で固定する）
    """

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        """クラスロジットを計算する．

        Args:
            x (torch.Tensor): 入力画像．形状は (N, 1, 28, 28) または (N, 784)．

        Returns:
            logits (torch.Tensor): クラスロジット．形状は (N, 10)．
        """
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits


def load_model(ModelClass, weight_path=None, seed=0):
    """モデルをインスタンス化し，必要なら学習済み重みを読み込む．

    Args:
        ModelClass (type): torch.nn.Module を継承したモデルクラス．
        weight_path (str | None): 学習済み重み (.pth) のパス．None なら初期値を使う．
        seed (int): パラメータ初期化用の乱数シード．

    Returns:
        model (torch.nn.Module): インスタンス化されたモデル．
    """
    set_seed(seed)
    model = ModelClass()
    if weight_path is not None:
        state_dict = torch.load(weight_path, map_location="cpu")
        model.load_state_dict(state_dict)
    return model


## 5. データセットと DataLoader

MNIST は公式の学習 60,000 枚 / テスト 10,000 枚に分割されている．画素値は `ToTensor()` により $[0, 1]$ に正規化する．DataLoader のシャッフルは `torch.Generator` で決定論的にする．


In [ ]:
def load_dataloader(seed=0, batch_size=128):
    """MNIST の学習用・検証用 DataLoader を作成する．

    Args:
        seed (int): データの並びを固定する乱数シード．
        batch_size (int): ミニバッチサイズ．

    Returns:
        train_dataloader (torch.utils.data.DataLoader): 学習用 DataLoader．
            各バッチは画像 (N, 1, 28, 28) とラベル (N,) のタプル．
        test_dataloader (torch.utils.data.DataLoader): 検証用 DataLoader．
            各バッチの形状は学習用と同じ．
    """
    set_seed(seed)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    transform = transforms.ToTensor()
    train_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=True,
        download=True,
        transform=transform,
    )
    test_dataset = datasets.MNIST(
        root=str(DATASET_DIR),
        train=False,
        download=True,
        transform=transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_dataloader, test_dataloader


### データの確認

学習用 DataLoader から 1 バッチを取り出し，画像の形状とラベルの例を表示する．


In [ ]:
preview_loader, _ = load_dataloader(seed=0, batch_size=8)
preview_images, preview_labels = next(iter(preview_loader))
print(f"images shape: {tuple(preview_images.shape)}")
print(f"labels shape: {tuple(preview_labels.shape)}")
print(f"labels      : {preview_labels.tolist()}")

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(preview_images[i, 0], cmap="gray")
    ax.set_title(f"label={preview_labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. 誤差関数と評価関数

分類の誤差は実験001と同じ交差エントロピーである．評価指標は Accuracy とする．


In [ ]:
def loss_func(outputs, teacher_signals):
    """モデル出力と教師信号の交差エントロピー誤差を計算する．

    Args:
        outputs (torch.Tensor): モデルのロジット．形状は (N, 10)．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)，dtype は long．

    Returns:
        loss (torch.Tensor): 1 データあたりの平均交差エントロピー．形状は ()．
    """
    loss = F.cross_entropy(outputs, teacher_signals, reduction="mean")
    return loss


def metrics_func(outputs, teacher_signals):
    """モデル出力と教師信号から Accuracy を計算する．

    Args:
        outputs (torch.Tensor): モデルのロジット．形状は (N, 10)．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)．

    Returns:
        metrics_to_value (dict): キー "acc" に 1 データあたりの正解率 (float) を格納した辞書．
    """
    pred_labels = outputs.argmax(dim=1)
    acc = (pred_labels == teacher_signals).float().mean().item()
    return {"acc": acc}


## 7. ノイズ最適化（敵対的摂動）

入力 $x$ に加えるノイズ $\delta$ を，交差エントロピーが大きくなる方向へ反復更新する．本実装は **PGD（Projected Gradient Descent，射影付き勾配法）** である．$L_{\infty}$ 球 $\lVert\delta\rVert_{\infty} \le \epsilon$ 内で多段攻撃を行う．各ステップは

$$
\delta \leftarrow \mathrm{clip}_{[-\epsilon,\epsilon]}\bigl(\delta + \alpha \,\mathrm{sign}(\nabla_{x+\delta} \mathcal{L})\bigr)
$$

であり，さらに $x+\delta$ が $[0, 1]$ に収まるよう射影する．$\epsilon=0.1$ はノイズの最大振幅，$\alpha=0.025$ は 1 ステップの更新幅，反復は 5 回である．

手順は次のとおりである．

1. $\delta \leftarrow 0$ から開始する
2. 固定したモデルで $x+\delta$ の交差エントロピー $\mathcal{L}$ を計算し，入力勾配を取る
3. $\mathrm{sign}(\nabla_{x+\delta}\mathcal{L})$ 方向へ $\alpha$ だけ更新する
4. 各画素を $[-\epsilon, \epsilon]$ にクリップする
5. $x+\delta \in [0,1]$ になるよう再射影する
6. 手順 2 から 5 を繰り返す

1 ステップのみなら **FGSM** に近い．反復と射影があるため PGD と呼ぶ．学習時は PGD で求めた $x+\delta$ を入力として更新する（**敵対的学習**）．


In [ ]:
def optimize_adversarial_noise(model, inputs, teacher_signals, epsilon, n_steps, step_size):
    """交差エントロピーを増やす方向へ入力ノイズを最適化する．

    Args:
        model (torch.nn.Module): 分類モデル．
        inputs (torch.Tensor): 入力画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)．
        epsilon (float): ノイズの L_inf 半径．
        n_steps (int): 勾配上昇の反復回数．
        step_size (float): 1 ステップの更新幅．

    Returns:
        noise (torch.Tensor): 最適化されたノイズ．形状は (N, 1, 28, 28)．
            各画素は [-epsilon, epsilon] に収まる．
    """
    was_training = model.training
    model.eval()

    noise = torch.zeros_like(inputs)
    for _ in range(n_steps):
        perturbed = (inputs + noise).clamp(0.0, 1.0).detach().requires_grad_(True)
        outputs = model(perturbed)
        attack_loss = loss_func(outputs, teacher_signals)
        grad = torch.autograd.grad(attack_loss, perturbed)[0]
        noise = noise + step_size * grad.sign()
        noise = noise.clamp(-epsilon, epsilon)
        noise = (inputs + noise).clamp(0.0, 1.0) - inputs

    if was_training:
        model.train()
    return noise.detach()


## 8. 学習ループ

学習は次の 3 段に分ける．実験001との違いは，`iteration` 内でノイズを最適化し，摂動画像 $x+\delta$ でパラメータを更新する点である．検証ではクリーン画像と敵対的画像の両方の Accuracy を返す．

1. `iteration`: ノイズ最適化 → 摂動画像での学習 / 検証
2. `epoch`: DataLoader 全件を処理し，1 データあたりの平均指標を返す
3. `train`: エポックを繰り返し，最良モデルとログを保存する


In [ ]:
def iteration(model, inputs, teacher_signals, optimizer=None):
    """1 ミニバッチを敵対的学習または検証する．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        inputs (torch.Tensor): 入力画像．形状は (N, 1, 28, 28)．
        teacher_signals (torch.Tensor): クラスラベル．形状は (N,)．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): "loss", "acc", "adv_acc" をキーとする平均値の辞書．
            loss は学習時は敵対的画像，検証時はクリーン画像の交差エントロピー．
    """
    noise = optimize_adversarial_noise(
        model,
        inputs,
        teacher_signals,
        epsilon=EPSILON,
        n_steps=ADV_STEPS,
        step_size=ADV_STEP_SIZE,
    )
    adv_inputs = (inputs + noise).clamp(0.0, 1.0)

    if optimizer is None:
        with torch.no_grad():
            clean_outputs = model(inputs)
            adv_outputs = model(adv_inputs)
            loss = loss_func(clean_outputs, teacher_signals)
    else:
        optimizer.zero_grad()
        adv_outputs = model(adv_inputs)
        loss = loss_func(adv_outputs, teacher_signals)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            clean_outputs = model(inputs)

    metrics_to_value = metrics_func(clean_outputs, teacher_signals)
    adv_metrics = metrics_func(adv_outputs, teacher_signals)
    metrics_to_value["acc"] = metrics_to_value["acc"]
    metrics_to_value["adv_acc"] = adv_metrics["acc"]
    metrics_to_value["loss"] = loss.item()
    return metrics_to_value


def epoch(model, dataloader, optimizer=None):
    """DataLoader 全件を 1 周し，1 データあたりの平均指標を返す．

    Args:
        model (torch.nn.Module): 学習 / 検証対象のモデル．
        dataloader (torch.utils.data.DataLoader): 入力とラベルの DataLoader．
        optimizer (torch.optim.Optimizer | None): 最適化手法．None なら検証モード．

    Returns:
        metrics_to_value (dict): エポック全体の 1 データあたり平均（"loss", "acc", "adv_acc"）．
    """
    if optimizer is None:
        model.eval()
    else:
        model.train()

    device = next(model.parameters()).device
    sum_metrics = {}
    n_total = 0

    progress = tqdm(dataloader, leave=False)
    for inputs, teacher_signals in progress:
        inputs = inputs.to(device)
        teacher_signals = teacher_signals.to(device)

        batch_metrics = iteration(model, inputs, teacher_signals, optimizer)
        batch_size = inputs.size(0)
        n_total += batch_size

        for name, value in batch_metrics.items():
            sum_metrics[name] = sum_metrics.get(name, 0.0) + value * batch_size

        average_metrics = {name: total / n_total for name, total in sum_metrics.items()}
        progress.set_postfix({name: f"{value:.4f}" for name, value in average_metrics.items()})

    metrics_to_value = {name: total / n_total for name, total in sum_metrics.items()}
    return metrics_to_value


In [ ]:
def build_optimizer(model, optimizer_name, lr):
    """最適化手法の名前から Optimizer を生成する．

    Args:
        model (torch.nn.Module): パラメータを更新するモデル．
        optimizer_name (str): 最適化手法名．"Adam" または "SGD"．
        lr (float): 学習率．

    Returns:
        optimizer (torch.optim.Optimizer): 生成された Optimizer．
    """
    if optimizer_name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=lr)
    if optimizer_name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"未対応の最適化手法である: {optimizer_name}")


def train(
    target_dir,
    ModelClass,
    load_dataloader,
    epochs,
    batch_size,
    seed=0,
    lr=0.001,
    optimizer_name="Adam",
):
    """モデルを敵対的学習し，最良重みとログを保存する．

    Args:
        target_dir (str): 結果の保存先ディレクトリ．
        ModelClass (type): 学習するモデルクラス．
        load_dataloader (callable): seed と batch_size から DataLoader を返す関数．
        epochs (int): 学習エポック数．
        batch_size (int): ミニバッチサイズ．
        seed (int): 乱数シード．
        lr (float): 学習率．
        optimizer_name (str): 最適化手法名．

    Returns:
        None
    """
    os.makedirs(target_dir, exist_ok=True)
    train_dataloader, test_dataloader = load_dataloader(seed=seed, batch_size=batch_size)
    model = load_model(ModelClass, weight_path=None, seed=seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = build_optimizer(model, optimizer_name, lr)

    logger = ResultLogger()
    logger.set_names(
        "train_loss",
        "train_acc",
        "train_adv_acc",
        "val_loss",
        "val_acc",
        "val_adv_acc",
    )

    train_metrics = epoch(model, train_dataloader, optimizer=None)
    val_metrics = epoch(model, test_dataloader, optimizer=None)
    logger(
        train_metrics["loss"],
        train_metrics["acc"],
        train_metrics["adv_acc"],
        val_metrics["loss"],
        val_metrics["acc"],
        val_metrics["adv_acc"],
    )
    print(
        f"[epoch 0/{epochs}] "
        f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} train_adv_acc={train_metrics['adv_acc']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} val_adv_acc={val_metrics['adv_acc']:.4f}"
    )

    best_val_adv_acc = val_metrics["adv_acc"]
    torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

    for epoch_idx in range(1, epochs + 1):
        train_metrics = epoch(model, train_dataloader, optimizer=optimizer)
        val_metrics = epoch(model, test_dataloader, optimizer=None)
        logger(
            train_metrics["loss"],
            train_metrics["acc"],
            train_metrics["adv_acc"],
            val_metrics["loss"],
            val_metrics["acc"],
            val_metrics["adv_acc"],
        )
        print(
            f"[epoch {epoch_idx}/{epochs}] "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} train_adv_acc={train_metrics['adv_acc']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} val_adv_acc={val_metrics['adv_acc']:.4f}"
        )

        if val_metrics["adv_acc"] > best_val_adv_acc:
            best_val_adv_acc = val_metrics["adv_acc"]
            torch.save(model.state_dict(), os.path.join(target_dir, "best_model.pth"))

        logger.save(os.path.join(target_dir, "log.json"))

    logger.save(os.path.join(target_dir, "log.json"))
    print(f"best val_adv_acc={best_val_adv_acc:.4f}")
    print(f"saved: {target_dir}")


## 9. 実験の実行


In [ ]:
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

for optimizer_name in OPTIMIZER_NAMES:
    for lr in LEARNING_RATES:
        for batch_size in BATCH_SIZES:
            for seed in SEEDS:
                hyperparam_name = f"{lr}_{batch_size}"
                target_dir = OUTPUT_ROOT / optimizer_name / hyperparam_name / str(seed)
                log_path = target_dir / "log.json"
                weight_path = target_dir / "best_model.pth"

                if log_path.exists() and weight_path.exists():
                    print(f"skip: {target_dir}")
                    continue

                print(f"train: {target_dir}")
                train(
                    target_dir=str(target_dir),
                    ModelClass=MNISTClassifier,
                    load_dataloader=load_dataloader,
                    epochs=EPOCHS,
                    batch_size=batch_size,
                    seed=seed,
                    lr=lr,
                    optimizer_name=optimizer_name,
                )


## 10. 学習曲線の確認

保存したログから Loss と，クリーン / 敵対的 Accuracy の推移を描画する．


In [ ]:
log_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "log.json"
logger = ResultLogger(str(log_path))
epochs_axis = list(range(len(logger["train_loss"])))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(epochs_axis, logger["train_loss"], label="train")
axes[0].plot(epochs_axis, logger["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Cross Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, logger["train_acc"], label="train")
axes[1].plot(epochs_axis, logger["val_acc"], label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Clean Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_axis, logger["train_adv_acc"], label="train")
axes[2].plot(epochs_axis, logger["val_adv_acc"], label="val")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("accuracy")
axes[2].set_title("Adversarial Accuracy")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"final train_acc     = {logger['train_acc'][-1]:.4f}")
print(f"final val_acc       = {logger['val_acc'][-1]:.4f}")
print(f"final train_adv_acc = {logger['train_adv_acc'][-1]:.4f}")
print(f"final val_adv_acc   = {logger['val_adv_acc'][-1]:.4f}")
print(f"best val_acc        = {max(logger['val_acc']):.4f}")
print(f"best val_adv_acc    = {max(logger['val_adv_acc']):.4f}")


## 11. 敵対的画像の確認

最良モデルで検証画像のノイズを最適化し，上段に入力，中段にノイズ（見やすいよう $[0,1]$ へ線形変換），下段に摂動画像を並べる．タイトルは正解ラベルと，クリーン / 敵対的入力での予測である．


In [ ]:
weight_path = OUTPUT_ROOT / OPTIMIZER_NAMES[0] / f"{LEARNING_RATES[0]}_{BATCH_SIZES[0]}" / str(SEEDS[0]) / "best_model.pth"
_, test_loader = load_dataloader(seed=SEEDS[0], batch_size=8)
images, labels = next(iter(test_loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(MNISTClassifier, weight_path=str(weight_path), seed=SEEDS[0])
model = model.to(device)
model.eval()

images = images.to(device)
labels = labels.to(device)
noise = optimize_adversarial_noise(
    model,
    images,
    labels,
    epsilon=EPSILON,
    n_steps=ADV_STEPS,
    step_size=ADV_STEP_SIZE,
)
adv_images = (images + noise).clamp(0.0, 1.0)

with torch.no_grad():
    clean_pred = model(images).argmax(dim=1).cpu()
    adv_pred = model(adv_images).argmax(dim=1).cpu()

images_cpu = images.cpu()
noise_cpu = noise.cpu()
adv_cpu = adv_images.cpu()
labels_cpu = labels.cpu()
noise_vis = (noise_cpu + EPSILON) / (2 * EPSILON)

fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for i in range(8):
    axes[0, i].imshow(images_cpu[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"y={labels_cpu[i].item()} pred={clean_pred[i].item()}")
    axes[0, i].axis("off")
    axes[1, i].imshow(noise_vis[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title("noise")
    axes[1, i].axis("off")
    axes[2, i].imshow(adv_cpu[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[2, i].set_title(f"adv pred={adv_pred[i].item()}")
    axes[2, i].axis("off")
plt.tight_layout()

figure_dir = OUTPUT_ROOT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
save_path = figure_dir / "adversarial_examples.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {save_path}")
print(f"clean pred: {clean_pred.tolist()}")
print(f"adv pred  : {adv_pred.tolist()}")
print(f"true      : {labels_cpu.tolist()}")
